#1. Project Set-up
##Installing required libraries and connecting to the data source

In [1]:
%pip -q install pyspark

###Creating the SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("HealthcareDataEngineeringPipeline").master("local[*]").getOrCreate())

print("Spark version:", spark.version)
print("Spark session created successfully!")

Spark version: 4.0.4
Spark session created successfully!


###Connecting to Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

file_path = ("/content/drive/MyDrive/"
            "healthcare-de-pipeline/"
            "healthcare_admissions_curated_5000.csv"
          )

if not Path(file_path).exists():
    raise FileNotFoundError(
        "File not found!"
    )

df = (spark.read.option("header", True).option("inferSchema", True).csv(file_path))
print("File loaded successfully!")
print("Total rows:", df.count())
print("Total columns:", len(df.columns))
df.printSchema()
df.show(5, truncate=False)


File loaded successfully!
Total rows: 5000
Total columns: 14
root
 |-- admission_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_type: string (nullable = true)
 |-- medical_condition: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- hospital: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- billing_amount: double (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- medication: string (nullable = true)
 |-- test_results: string (nullable = true)
 |-- patient_id: string (nullable = true)

+---------------+---+------+----------+-----------------+--------------+----------------------------+------------------+--------------+--------------+--------------+-----------+------------+----------+
|admission_id   |age|gender|blood_type|medical_condition|admission_date|hospital                    |insurance_provider|bill

###Authenticating and reading Google Sheets

In [5]:
from google.colab import auth
import google.auth

auth.authenticate_user()
credentials, project_id = google.auth.default()

print("Google authorization completed.")


Google authorization completed.


In [6]:
%pip -q install gspread

In [7]:
import gspread

gc = gspread.authorize(credentials)


SPREADSHEET_ID = "1fRryox9nK284M1Lbi8D5rco3-64Ne4cCpgC_qDV8kaM"

spreadsheet = gc.open_by_key(SPREADSHEET_ID)
worksheet = spreadsheet.worksheet("incoming_changes")

sheet_values = worksheet.get_all_values()

if not sheet_values:
    raise ValueError("The Google Sheet is empty.")

headers = sheet_values[0]
data_rows = sheet_values[1:]

print("Sheet connected:", spreadsheet.title)
print("Worksheet:", worksheet.title)
print("Incoming data rows:", len(data_rows))
print("Columns:", headers)

incoming_df = spark.createDataFrame(data_rows, schema=headers)

print("Spark DataFrame created successfully.")
incoming_df.show(5, truncate=False)

Sheet connected: healthcare_incoming_changes
Worksheet: incoming_changes
Incoming data rows: 100
Columns: ['admission_id', 'age', 'gender', 'blood_type', 'medical_condition', 'admission_date', 'hospital', 'insurance_provider', 'billing_amount', 'admission_type', 'discharge_date', 'medication', 'test_results', 'patient_id', 'source_change_type']
Spark DataFrame created successfully.
+---------------+---+------+----------+-----------------+--------------+----------------------------+------------------+--------------+--------------+--------------+----------+------------+----------+------------------+
|admission_id   |age|gender|blood_type|medical_condition|admission_date|hospital                    |insurance_provider|billing_amount|admission_type|discharge_date|medication|test_results|patient_id|source_change_type|
+---------------+---+------+----------+-----------------+--------------+----------------------------+------------------+--------------+--------------+--------------+----------

#2. Bronze Layer
##2.1 Raw Data storage

In [8]:
# Bronze layer: preserve the source DataFrames as read
# df comes from the historical CSV
# incoming_df comes from Google Sheets

initial_bronze_df = df
incoming_bronze_df = incoming_df

print("BRONZE LAYER SOURCE SUMMARY")
print("-" * 40)

print("Historical source rows:", initial_bronze_df.count())
print("Historical source columns:", len(initial_bronze_df.columns))

print("Incoming source rows:", incoming_bronze_df.count())
print("Incoming source columns:", len(incoming_bronze_df.columns))

print("\nIncoming change types:")
incoming_bronze_df.groupBy("source_change_type").count().show()

print("\nHistorical source sample:")
initial_bronze_df.show(3, truncate=False)

print("\nIncoming source sample:")
incoming_bronze_df.show(3, truncate=False)


BRONZE LAYER SOURCE SUMMARY
----------------------------------------
Historical source rows: 5000
Historical source columns: 14
Incoming source rows: 100
Incoming source columns: 15

Incoming change types:
+------------------+-----+
|source_change_type|count|
+------------------+-----+
|            UPDATE|   50|
|            INSERT|   50|
+------------------+-----+


Historical source sample:
+---------------+---+------+----------+-----------------+--------------+--------------+------------------+--------------+--------------+--------------+-----------+------------+----------+
|admission_id   |age|gender|blood_type|medical_condition|admission_date|hospital      |insurance_provider|billing_amount|admission_type|discharge_date|medication |test_results|patient_id|
+---------------+---+------+----------+-----------------+--------------+--------------+------------------+--------------+--------------+--------------+-----------+------------+----------+
|ADMBE2577CEA03A|49 |Male  |B+        |O

In [9]:
import os
import shutil

bronze_path = "/content/drive/MyDrive/healthcare-de-pipeline/bronze"

initial_path = bronze_path + "/initial_snapshot"
batch_path = bronze_path + "/incoming_changes_snapshot"

# Save historical data
if os.path.exists(initial_path):
    shutil.rmtree(initial_path)

df.write.mode("overwrite").parquet(initial_path)

# Save incoming changes
if os.path.exists(batch_path):
    shutil.rmtree(batch_path)

incoming_df.write.mode("overwrite").parquet(batch_path)

print("Bronze snapshots saved.")

Bronze snapshots saved.


In [10]:
# Read the saved snapshots
raw_df = spark.read.parquet(initial_path)
batch_df = spark.read.parquet(batch_path)

# Check row counts
raw_count = raw_df.count()
batch_count = batch_df.count()

print("Historical rows:", raw_count)
print("Incoming rows:", batch_count)

assert raw_count == 5000, "Historical row count mismatch"
assert batch_count == 100, "Incoming row count mismatch"
assert "source_change_type" in batch_df.columns

print("Bronze verification passed.")


Historical rows: 5000
Incoming rows: 100
Bronze verification passed.


# 3. Silver Layer: Data Cleaning and Validation
## 3.1 Inspect Bronze Data

In [11]:
# Check the historical data
print("HISTORICAL DATA")
print("Rows:", raw_df.count())
raw_df.printSchema()

print("\nHISTORICAL SAMPLE")
raw_df.show(5, truncate=False)

# Check the incoming batch
print("\nINCOMING DATA")
print("Rows:", batch_df.count())
batch_df.printSchema()

print("\nINCOMING CHANGE TYPES")
batch_df.groupBy("source_change_type").count().show()

print("\nINCOMING SAMPLE")
batch_df.show(5, truncate=False)

HISTORICAL DATA
Rows: 5000
root
 |-- admission_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_type: string (nullable = true)
 |-- medical_condition: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- hospital: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- billing_amount: double (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- medication: string (nullable = true)
 |-- test_results: string (nullable = true)
 |-- patient_id: string (nullable = true)


HISTORICAL SAMPLE
+---------------+---+------+----------+-----------------+--------------+----------------------------+------------------+--------------+--------------+--------------+-----------+------------+----------+
|admission_id   |age|gender|blood_type|medical_condition|admission_date|hospital                    |insurance_provider|billing_amount|admi

##3.2 Standardize Columns and Data Types

In [12]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

def prep(df):
    #Remove extra spaces from text columns
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                field.name,
                F.trim(F.col(field.name))
            )

    #Set correct data types
    df = (df.withColumn("age", F.col("age").cast("int")).withColumn("billing_amount",F.col("billing_amount").cast("double"))
          .withColumn("admission_date",F.to_date("admission_date", "yyyy-MM-dd"))
          .withColumn("discharge_date",F.to_date("discharge_date", "yyyy-MM-dd"))
      )

    #Keep change type consistent
    if "source_change_type" in df.columns:
        df = df.withColumn("source_change_type",F.upper(F.col("source_change_type"))
            )
    return df

# Prepare both datasets
clean_df = prep(raw_df)
batch_clean = prep(batch_df)

print("Historical schema:")
clean_df.printSchema()

print("\nIncoming schema:")
batch_clean.printSchema()

print("\nHistorical sample:")
clean_df.show(5, truncate=False)

print("\nIncoming sample:")
batch_clean.show(5, truncate=False)

Historical schema:
root
 |-- admission_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_type: string (nullable = true)
 |-- medical_condition: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- hospital: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- billing_amount: double (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- medication: string (nullable = true)
 |-- test_results: string (nullable = true)
 |-- patient_id: string (nullable = true)


Incoming schema:
root
 |-- admission_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_type: string (nullable = true)
 |-- medical_condition: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- hospital: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |

In [13]:
print("Historical row count:", clean_df.count())
print("Incoming row count:", batch_clean.count())
print("\nHistorical data types:")
clean_df.select("age", "billing_amount","admission_date", "discharge_date").printSchema()
print("\nIncoming change types:")
batch_clean.groupBy("source_change_type").count().show()

Historical row count: 5000
Incoming row count: 100

Historical data types:
root
 |-- age: integer (nullable = true)
 |-- billing_amount: double (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)


Incoming change types:
+------------------+-----+
|source_change_type|count|
+------------------+-----+
|            UPDATE|   50|
|            INSERT|   50|
+------------------+-----+



##3.3 Data Quality Checks

In [14]:
from pyspark.sql import functions as F
# Columns that should not be missing
req_cols = ["admission_id","admission_date","patient_id","age","billing_amount"]

def check(df, name):
    print(f"\n{name}")
    print("-" * 35)
    print("Rows:", df.count())

    # Missing values in required columns
    print("\nMissing required values:")
    df.select([
        F.sum(
            F.when(
                F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in req_cols
    ]).show()

    # Invalid ages and billing amounts
    print("Invalid age:", df.filter(
        (F.col("age") < 0) | (F.col("age") > 120)
    ).count())

    print("Negative billing:", df.filter(
        F.col("billing_amount") < 0
    ).count())

    # Invalid admission/discharge date order
    print("Discharge before admission:", df.filter(
        F.col("discharge_date") < F.col("admission_date")
    ).count())

    # Duplicate admission IDs
    print("Duplicate admission IDs:", df.count() - df.select(
        "admission_id"
    ).distinct().count())

check(clean_df, "HISTORICAL DATA")
check(batch_clean, "INCOMING BATCH")


HISTORICAL DATA
-----------------------------------
Rows: 5000

Missing required values:
+------------+--------------+----------+---+--------------+
|admission_id|admission_date|patient_id|age|billing_amount|
+------------+--------------+----------+---+--------------+
|           0|             0|         0|  0|             0|
+------------+--------------+----------+---+--------------+

Invalid age: 0
Negative billing: 0
Discharge before admission: 0
Duplicate admission IDs: 0

INCOMING BATCH
-----------------------------------
Rows: 100

Missing required values:
+------------+--------------+----------+---+--------------+
|admission_id|admission_date|patient_id|age|billing_amount|
+------------+--------------+----------+---+--------------+
|           0|             0|         0|  0|             0|
+------------+--------------+----------+---+--------------+

Invalid age: 0
Negative billing: 0
Discharge before admission: 10
Duplicate admission IDs: 0


###Handling invalid records

In [15]:
bad_dates = batch_clean.filter(
    F.col("discharge_date") < F.col("admission_date")
)

bad_dates.select(
    "admission_id",
    "source_change_type",
    "admission_date",
    "discharge_date"
).show(20, truncate=False)

print("Invalid records:", bad_dates.count())

bad_dates.groupBy("source_change_type").count().show()

+------------+------------------+--------------+--------------+
|admission_id|source_change_type|admission_date|discharge_date|
+------------+------------------+--------------+--------------+
|ADMNEW00007 |INSERT            |2023-11-28    |2023-11-27    |
|ADMNEW00009 |INSERT            |2022-01-31    |2022-01-30    |
|ADMNEW00017 |INSERT            |2023-10-26    |2023-10-15    |
|ADMNEW00023 |INSERT            |2021-08-18    |2021-08-13    |
|ADMNEW00026 |INSERT            |2020-06-08    |2020-06-05    |
|ADMNEW00029 |INSERT            |2020-11-03    |2020-11-02    |
|ADMNEW00039 |INSERT            |2020-09-22    |2020-09-12    |
|ADMNEW00040 |INSERT            |2021-01-03    |2020-12-25    |
|ADMNEW00042 |INSERT            |2022-12-02    |2022-11-21    |
|ADMNEW00043 |INSERT            |2024-02-04    |2024-02-03    |
+------------+------------------+--------------+--------------+

Invalid records: 10
+------------------+-----+
|source_change_type|count|
+------------------+-----+
| 

In [16]:
# Separate invalid incoming records
bad_dates = batch_clean.filter(
    F.col("discharge_date") < F.col("admission_date")
)

# Keep records that pass the date check
batch_clean = batch_clean.filter(
    F.col("discharge_date").isNull()
    | (F.col("discharge_date") >= F.col("admission_date"))
)

print("Historical rows:", clean_df.count())
print("Valid incoming rows:", batch_clean.count())
print("Quarantined rows:", bad_dates.count())

print("\nValid incoming change types:")
batch_clean.groupBy("source_change_type").count().show()

Historical rows: 5000
Valid incoming rows: 90
Quarantined rows: 10

Valid incoming change types:
+------------------+-----+
|source_change_type|count|
+------------------+-----+
|            UPDATE|   50|
|            INSERT|   40|
+------------------+-----+



In [17]:
# Check duplicate IDs in historical data
print("Historical duplicate IDs:",
      clean_df.count() - clean_df.select("admission_id").distinct().count())

# Check duplicate IDs in the valid incoming batch
print("Incoming duplicate IDs:",
      batch_clean.count() - batch_clean.select("admission_id").distinct().count())

# Check incoming operation labels
batch_clean.groupBy("source_change_type").count().show()

# Check whether updates refer to existing historical admissions
updates = batch_clean.filter(
    F.col("source_change_type") == "UPDATE"
)

new_rows = batch_clean.filter(
    F.col("source_change_type") == "INSERT"
)

missing_updates = updates.join(
    clean_df.select("admission_id"),
    on="admission_id",
    how="left_anti"
)

existing_inserts = new_rows.join(
    clean_df.select("admission_id"),
    on="admission_id",
    how="inner"
)

print("Updates without a historical match:", missing_updates.count())
print("Inserts that already exist historically:", existing_inserts.count())

Historical duplicate IDs: 0
Incoming duplicate IDs: 0
+------------------+-----+
|source_change_type|count|
+------------------+-----+
|            UPDATE|   50|
|            INSERT|   40|
+------------------+-----+

Updates without a historical match: 0
Inserts that already exist historically: 0


##3.4 Save Silver Data and Quarantine Invalid Records

In [18]:
import os
silver_path = "/content/drive/MyDrive/healthcare-de-pipeline/silver"
bad_path = "/content/drive/MyDrive/healthcare-de-pipeline/quarantine"

# Create folders in Drive
os.makedirs(silver_path, exist_ok=True)
os.makedirs(bad_path, exist_ok=True)

# Save cleaned historical data
clean_df.write.mode("overwrite").parquet(silver_path + "/historical")

# Save valid incoming changes
batch_clean.write.mode("overwrite").parquet(silver_path + "/incoming")

# Save invalid records for investigation
bad_dates.write.mode("overwrite").parquet(bad_path + "/invalid_dates")
print("Silver data and quarantine records saved.")

Silver data and quarantine records saved.


In [19]:
# Read saved data back from Drive
silver_df = spark.read.parquet(silver_path + "/historical")
silver_batch = spark.read.parquet(silver_path + "/incoming")
quarantine_df = spark.read.parquet(bad_path + "/invalid_dates")

print("Historical Silver rows:", silver_df.count())
print("Incoming Silver rows:", silver_batch.count())
print("Quarantined rows:", quarantine_df.count())

print("\nIncoming change types:")
silver_batch.groupBy("source_change_type").count().show()

assert silver_df.count() == 5000
assert silver_batch.count() == 90
assert quarantine_df.count() == 10

print("Silver persistence verification passed.")

Historical Silver rows: 5000
Incoming Silver rows: 90
Quarantined rows: 10

Incoming change types:
+------------------+-----+
|source_change_type|count|
+------------------+-----+
|            UPDATE|   50|
|            INSERT|   40|
+------------------+-----+

Silver persistence verification passed.


#4. Setting up a database
##4.1 Using Neon PostgreSQL

In [20]:
%pip -q install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 46.1 MB/s eta 0:00:00


In [33]:
from google.colab import userdata
import psycopg2

db_url = userdata.get("NEON_DATABASE_URL")

conn = psycopg2.connect(db_url)
cur = conn.cursor()

print("Connected to PostgreSQL successfully.")

Connected to PostgreSQL successfully.


##4.2 Creating the Table

In [34]:
cur.execute("""
CREATE TABLE IF NOT EXISTS admissions (
    admission_id VARCHAR(30) PRIMARY KEY,
    age INTEGER,
    gender VARCHAR(10),
    blood_type VARCHAR(4),
    medical_condition VARCHAR(100),
    admission_date DATE,
    hospital VARCHAR(200),
    insurance_provider VARCHAR(100),
    billing_amount NUMERIC(12, 2),
    admission_type VARCHAR(50),
    discharge_date DATE,
    medication VARCHAR(100),
    test_results VARCHAR(50),
    patient_id VARCHAR(30)
);
""")

conn.commit()

print("Admissions table created successfully.")

Admissions table created successfully.


##4.3 Loading the data

In [35]:
#Reading the saved Silver tier data
silver_df = spark.read.parquet(silver_path + "/historical")

print("Historical records:", silver_df.count())
print("Columns:", silver_df.columns)

Historical records: 5000
Columns: ['admission_id', 'age', 'gender', 'blood_type', 'medical_condition', 'admission_date', 'hospital', 'insurance_provider', 'billing_amount', 'admission_type', 'discharge_date', 'medication', 'test_results', 'patient_id']


In [36]:
from psycopg2.extras import execute_values

cols = [
    "admission_id", "age", "gender", "blood_type",
    "medical_condition", "admission_date", "hospital",
    "insurance_provider", "billing_amount", "admission_type",
    "discharge_date", "medication", "test_results", "patient_id"
]

rows = [
    tuple(row)
    for row in silver_df.select(*cols).collect()
]

cur = conn.cursor()

sql = """
INSERT INTO admissions (
    admission_id, age, gender, blood_type, medical_condition,
    admission_date, hospital, insurance_provider, billing_amount,
    admission_type, discharge_date, medication, test_results, patient_id
)
VALUES %s
ON CONFLICT (admission_id) DO NOTHING
"""

execute_values(cur, sql, rows, page_size=500)
conn.commit()

print("Historical data loaded.")

Historical data loaded.


In [37]:
#Verifying the database
cur.execute("SELECT COUNT(*) FROM admissions")
print("Records in PostgreSQL:", cur.fetchone()[0])

Records in PostgreSQL: 5000


##Loading the incoming batch

In [38]:
silver_batch = spark.read.parquet(silver_path + "/incoming")

print("Incoming records:", silver_batch.count())
silver_batch.groupBy("source_change_type").count().show()

Incoming records: 90
+------------------+-----+
|source_change_type|count|
+------------------+-----+
|            UPDATE|   50|
|            INSERT|   40|
+------------------+-----+



##UPSERT the Records

In [39]:
from psycopg2.extras import execute_values

cols = [
    "admission_id", "age", "gender", "blood_type",
    "medical_condition", "admission_date", "hospital",
    "insurance_provider", "billing_amount", "admission_type",
    "discharge_date", "medication", "test_results", "patient_id"
]

rows = [
    tuple(row)
    for row in silver_batch.select(*cols).collect()
]

sql = """
INSERT INTO admissions (
    admission_id, age, gender, blood_type, medical_condition,
    admission_date, hospital, insurance_provider, billing_amount,
    admission_type, discharge_date, medication, test_results, patient_id
)
VALUES %s
ON CONFLICT (admission_id) DO UPDATE SET
    age = EXCLUDED.age,
    gender = EXCLUDED.gender,
    blood_type = EXCLUDED.blood_type,
    medical_condition = EXCLUDED.medical_condition,
    admission_date = EXCLUDED.admission_date,
    hospital = EXCLUDED.hospital,
    insurance_provider = EXCLUDED.insurance_provider,
    billing_amount = EXCLUDED.billing_amount,
    admission_type = EXCLUDED.admission_type,
    discharge_date = EXCLUDED.discharge_date,
    medication = EXCLUDED.medication,
    test_results = EXCLUDED.test_results,
    patient_id = EXCLUDED.patient_id
"""

cur = conn.cursor()
execute_values(cur, sql, rows, page_size=100)
conn.commit()

print("Incremental upsert completed.")

Incremental upsert completed.


##Verifying the final record

In [40]:
cur.execute("SELECT COUNT(*) FROM admissions")
print("Final records:", cur.fetchone()[0])

cur.execute("""
SELECT COUNT(*)
FROM admissions
WHERE admission_id LIKE 'ADMNEW%'
""")
print("New records:", cur.fetchone()[0])

Final records: 5040
New records: 40


##5. The Gold analytics

##5.1 Three analytical queries

In [41]:
cur = conn.cursor()

# 1. Admissions by medical condition
cur.execute("""
SELECT medical_condition, COUNT(*) AS total_admissions
FROM admissions
GROUP BY medical_condition
ORDER BY total_admissions DESC;
""")

print("Admissions by medical condition:")
for row in cur.fetchall():
    print(row)


# 2. Average billing by admission type
cur.execute("""
SELECT admission_type,
       COUNT(*) AS total_admissions,
       ROUND(AVG(billing_amount), 2) AS average_billing
FROM admissions
GROUP BY admission_type
ORDER BY average_billing DESC;
""")

print("\nAverage billing by admission type:")
for row in cur.fetchall():
    print(row)


# 3. Monthly admissions
cur.execute("""
SELECT DATE_TRUNC('month', admission_date)::date AS admission_month,
       COUNT(*) AS total_admissions
FROM admissions
GROUP BY DATE_TRUNC('month', admission_date)
ORDER BY admission_month;
""")

print("\nMonthly admissions:")
for row in cur.fetchall():
    print(row)

cur.close()

Admissions by medical condition:
('Asthma', 874)
('Diabetes', 864)
('Arthritis', 855)
('Cancer', 845)
('Hypertension', 820)
('Obesity', 782)

Average billing by admission type:
('Urgent', 1710, Decimal('25871.79'))
('Elective', 1687, Decimal('25827.60'))
('Emergency', 1643, Decimal('25166.27'))

Monthly admissions:
(datetime.date(2019, 5, 1), 64)
(datetime.date(2019, 6, 1), 88)
(datetime.date(2019, 7, 1), 81)
(datetime.date(2019, 8, 1), 85)
(datetime.date(2019, 9, 1), 89)
(datetime.date(2019, 10, 1), 101)
(datetime.date(2019, 11, 1), 85)
(datetime.date(2019, 12, 1), 91)
(datetime.date(2020, 1, 1), 97)
(datetime.date(2020, 2, 1), 91)
(datetime.date(2020, 3, 1), 67)
(datetime.date(2020, 4, 1), 83)
(datetime.date(2020, 5, 1), 93)
(datetime.date(2020, 6, 1), 79)
(datetime.date(2020, 7, 1), 97)
(datetime.date(2020, 8, 1), 80)
(datetime.date(2020, 9, 1), 72)
(datetime.date(2020, 10, 1), 84)
(datetime.date(2020, 11, 1), 92)
(datetime.date(2020, 12, 1), 100)
(datetime.date(2021, 1, 1), 92)
(da

##6. Validating the final database

In [42]:
cur = conn.cursor()

cur.execute("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT admission_id) AS unique_ids,
    SUM(CASE
        WHEN discharge_date < admission_date THEN 1
        ELSE 0
    END) AS invalid_dates
FROM admissions;
""")

print("Final validation:", cur.fetchone())

cur.execute("""
SELECT COUNT(*)
FROM admissions
WHERE admission_id LIKE 'ADMNEW%';
""")

print("New records:", cur.fetchone()[0])

cur.close()

Final validation: (5040, 5040, 0)
New records: 40
